# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [2]:
import os, getpass
import numpy as np
import pandas as pd
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

RANDOM_STATE = 42

In [3]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                            AS imp_h1,
        SUM(gsc_clicks)                                                 AS clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)               AS ctr_h1,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0)       AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_h1
    FROM {fact_march}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

label_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31') AS imp_h2
    FROM {fact_march}
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(label_frame, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining_proxy"] = (data["imp_h2"] < 0.8 * data["imp_h1"]).astype(int)
data = data.dropna(subset=["ctr_h1", "avg_position_h1"]).reset_index(drop=True)


def position_tier(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"

data["position_tier"] = data["avg_position_h1"].apply(position_tier)
data["is_quick_win_zone"] = data["position_tier"] == "11-20"
ctr_median = data.loc[data["position_tier"].isin(["1-3", "4-10"]), "ctr_h1"].median()
data["is_ctr_fix_zone"] = data["position_tier"].isin(["1-3", "4-10"]) & (data["ctr_h1"] < ctr_median)

def baseline_score_row(r):
    if r["is_quick_win_zone"]:
        return r["imp_h1"]
    if r["is_ctr_fix_zone"]:
        return r["imp_h1"] * (1 - r["ctr_h1"])
    return r["imp_h1"] * 0.1

data["baseline_score"] = data.apply(baseline_score_row, axis=1)

print(f"{len(data):,} rows | is_declining_proxy rate: {data['is_declining_proxy'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,475 rows | is_declining_proxy rate: 0.296


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


This is still a "yes/no with an observed label" question
(`is_declining_proxy`), so the menu is Logistic Regression (readable) -> Decision Tree (still
readable, catches interactions) -> Random Forest (stronger, only kept if it earns the drop in
readability). Gradient Boosting is skipped unless Random Forest's margin over the Decision Tree
turns out too small to justify stopping there -- see the check-in cell right after the
comparison table.

Clustering isn't used, for the same reason as before: this is a yes/no prediction with an
observed label, not a grouping question.

Note one real difference from the Week 2 framing: `is_declining_proxy` is a *tighter*, more
short-term signal than the CSV lane's `is_declining_label` (two halves of one month vs. a full
90-day trend), so don't expect the exact same numbers as the old `outputs/model_report.md` --
that comparison isn't apples-to-apples anymore, and that's fine; the Week 4 baseline above is
the one that matters now.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



*Grouped by `client_hash_id`, because of : rows from the same client share
conventions and baseline traffic levels, so a random row split would leak client identity into
the test set. Client-holdout is the honest split for "does this generalize to a client the
model hasn't seen."*

In [4]:
def client_holdout_split(frame: pd.DataFrame):
    all_indices = np.arange(len(frame))
    client_series = frame["client_hash_id"].astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()

    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(unique_clients)
    n_test = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:n_test])

    test_mask = client_series.isin(test_clients).to_numpy()
    return all_indices[~test_mask], all_indices[test_mask]

train_idx, test_idx = client_holdout_split(data)
overlap = set(data.iloc[train_idx]["client_hash_id"]) & set(data.iloc[test_idx]["client_hash_id"])

print(f"train rows: {len(train_idx):,} | test rows: {len(test_idx):,}")
print(f"clients present in BOTH splits: {len(overlap)} (should be 0)")
print(f"test-split decline rate: {data.iloc[test_idx]['is_declining_proxy'].mean():.3f} "
      f"(train: {data.iloc[train_idx]['is_declining_proxy'].mean():.3f})")

train rows: 92,659 | test rows: 27,816
clients present in BOTH splits: 0 (should be 0)
test-split decline rate: 0.295 (train: 0.297)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

FEATURES = ["imp_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]

X_all = data[FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
y_all = data["is_declining_proxy"].astype(int)

X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def score_row(y_true, scores, label):
    return {
        "model": label,
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "precision_at_100": precision_at_k(y_true, scores, 100),
        "roc_auc": roc_auc_score(y_true, scores),
        "avg_precision": average_precision_score(y_true, scores),
    }

results = [{
    "model": "base_rate (random order)",
    "precision_at_20": y_test.mean(), "precision_at_50": y_test.mean(),
    "precision_at_100": y_test.mean(), "roc_auc": 0.5, "avg_precision": y_test.mean(),
}]

baseline_test_scores = data.iloc[test_idx]["baseline_score"].to_numpy()
results.append(score_row(y_test, baseline_test_scores, "baseline_rules"))

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
    ),
}

fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_test)[:, 1]
    results.append(score_row(y_test, proba, name))

comparison = pd.DataFrame(results).set_index("model").round(3)
comparison

,precision_at_20,precision_at_50,precision_at_100,roc_auc,avg_precision
model,,,,,
base_rate (random order),0.295,0.295,0.295,0.500,0.295
baseline_rules,0.450,0.380,0.320,0.515,0.305
logistic_regression,0.750,0.700,0.720,0.545,0.342
decision_tree,0.250,0.500,0.570,0.612,0.381
random_forest,0.650,0.660,0.620,0.629,0.416


In [7]:
# Gradient Boosting check-in from Section 1.
rf_gain = comparison.loc["random_forest", "precision_at_50"] - comparison.loc["decision_tree", "precision_at_50"]
print(f"Random Forest vs Decision Tree gap on Precision@50: {rf_gain:.3f}")
print("Random Forest clearly beats the Decision Tree, but Logistic Regression beats")
print("BOTH of them (0.700 vs RF's 0.660 vs DT's 0.500) -- so the added complexity of")
print("Random Forest doesn't actually earn its keep here. Logistic Regression is the")
print("best model on this data, and that's the one Section 4's error analysis uses.")

Random Forest vs Decision Tree gap on Precision@50: 0.160
Random Forest clearly beats the Decision Tree, but Logistic Regression beats
BOTH of them (0.700 vs RF's 0.660 vs DT's 0.500) -- so the added complexity of
Random Forest doesn't actually earn its keep here. Logistic Regression is the
best model on this data, and that's the one Section 4's error analysis uses.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [8]:
best_model_name = comparison.drop(index=["base_rate (random order)", "baseline_rules"])["precision_at_50"].idxmax()
best_model = fitted[best_model_name]
print(f"Best model by Precision@50: {best_model_name}")

test_frame = data.iloc[test_idx].copy().reset_index(drop=True)
test_frame["predicted_proba"] = best_model.predict_proba(X_test)[:, 1]
test_frame["predicted_label"] = (test_frame["predicted_proba"] >= 0.5).astype(int)
test_frame["correct"] = test_frame["predicted_label"] == test_frame["is_declining_proxy"]
print(f"Overall test accuracy: {test_frame['correct'].mean():.3f}")

Best model by Precision@50: logistic_regression
Overall test accuracy: 0.458


In [11]:
by_position = test_frame.groupby("position_tier")["correct"].agg(["mean", "count"]).rename(columns={"mean": "accuracy"})
print("Accuracy by position tier:")
print(by_position.sort_values("accuracy"))
print("Accuracy is weakest across 1-3, 4-10, and 11-20 (all ~0.41-0.43) and clearly best")
print("on 21+ (0.545) -- the model struggles fairly uniformly across every better-ranked")
print("tier, not one specific one. Overall accuracy (0.458) looks low, but that's expected:")
print("class_weight='balanced' trades accuracy for better recall on the minority")
print("(declining) class, which matters more for a review queue than raw accuracy does.")


Accuracy by position tier:
               accuracy  count
position_tier                 
1-3            0.408828   1903
4-10           0.415911  10810
11-20          0.425493   6496
21+            0.544905   8607
Accuracy is weakest across 1-3, 4-10, and 11-20 (all ~0.41-0.43) and clearly best
on 21+ (0.545) -- the model struggles fairly uniformly across every better-ranked
tier, not one specific one. Overall accuracy (0.458) looks low, but that's expected:
class_weight='balanced' trades accuracy for better recall on the minority
(declining) class, which matters more for a review queue than raw accuracy does.


In [12]:
wrong = test_frame[~test_frame["correct"]].copy()
wrong["confidence_gap"] = (wrong["predicted_proba"] - 0.5).abs()
hard_cases = wrong.sort_values("confidence_gap").head(3)
cols = ["content_hash_id", "is_declining_proxy", "predicted_label", "predicted_proba",
        "avg_position_h1", "imp_h1", "active_days_h1"]
hard_cases[cols]
print("All three misses sit right at the model's 0.5 decision boundary rather than being")
print("confidently wrong -- the model was genuinely unsure on these, not fooled. That's the")
print("more forgivable kind of error: Precision@50 barely notices these, since they're not")
print("high-confidence top-ranked picks, they'd only count as mistakes under strict accuracy.")

All three misses sit right at the model's 0.5 decision boundary rather than being
confidently wrong -- the model was genuinely unsure on these, not fooled. That's the
more forgivable kind of error: Precision@50 barely notices these, since they're not
high-confidence top-ranked picks, they'd only count as mistakes under strict accuracy.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.